In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [8]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,"description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch": {"price": 199.99,"description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00,"description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand": {"price": 34.99,"description": "Adjustable aluminum, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())

    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"

    return str(p)

llm=ChatGroq(model="openai/gpt-oss-120b",temperature=0)

agent=create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store."
)

In [12]:
def ask(question:str):
    result=agent.invoke({"messages":[{"role":"user","content":question}]})
    print(result["messages"][-1].content)

In [13]:
ask("what is the price of the wireless headphones.")

The wireless headphones are priced at **$79.99**.


In [15]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262,"rating": 4.6},
    "smart watch": {"reviews": 340,"rating": 3.9},
    "mechanical keyboard": {"reviews": 67,"rating": 4.8},
    "laptop stand": {"reviews": 781,"rating": 4.5},
}

@tool
def get_review(name:str)->str:
    """Look up a product review by a product name.Return the product name,number of reviews and rating."""
    r=REVIEWS.get(name.lower())
    if not r:
        return f"Review not avialable for this product"
    return str(r)

In [26]:
llm=ChatGroq(model="openai/gpt-oss-120b",temperature=0)

agent2=create_agent(
    llm,
    tools=[get_product,get_review],
    system_prompt="You are a helpful product assistant for an online tech store."
)

def ask2(question:str):
    result=agent2.invoke({"messages":[{"role":"user","content":question}]})
    print(result["messages"][-1].content)

In [27]:
ask2("how do people like smart watch??")

People’s overall impression of the **smart watch** is fairly positive, though not overwhelmingly enthusiastic.  

- **Average rating:** **3.9 / 5**  
- **Number of reviews:** **340**  

A 3.9‑star score suggests that most owners are satisfied with the device’s core features—such as fitness tracking, notifications, and battery life—but there are a few common pain points that keep the rating from being higher. Typical feedback includes:

| What users like | What users wish were better |
|----------------|-----------------------------|
| Accurate health & activity tracking (heart‑rate, steps, sleep) | More robust app ecosystem and better third‑party integration |
| Convenient notification handling (calls, texts, calendar) | Slightly larger or more customizable display |
| Comfortable fit and decent battery life (1–2 days) | Improved durability (scratch‑resistant screen) |
| Stylish design options | Faster charging and longer battery endurance |

Overall, the smart watch is well‑received f

In [28]:
ask2("what is the price and reviews of smart watch?")

**Smart Watch**  
- **Price:** $199.99  
- **Average Rating:** 3.9 ★ (based on 340 reviews)  

If you’d like more details or want to see similar products, just let me know!


In [34]:
from langgraph.checkpoint.memory import InMemorySaver

llm=ChatGroq(model="openai/gpt-oss-120b",temperature=0)

agent2=create_agent(
    llm,
    tools=[get_product,get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question:str):
    config={"configurable":{"thread_id":"user-alice-session-1"}}
    result=agent2.invoke({"messages":[{"role":"user","content":question}]},config)
    print(result["messages"][-1].content)


In [35]:
ask2("what is the price wireless headphone?")

The wireless headphones are priced at **$79.99**.


In [36]:
ask2("what are the reviews on this product?")

The wireless headphones have **1,262 reviews** and an average rating of **4.6 / 5**.
